<a href="https://colab.research.google.com/github/sumitsingh63/EAA_Project/blob/main/03_statistical_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Statistical Analysis

## Panchang-Based Astronomical Indicators Against Historical Weather Trends

### Objective

This notebook statistically evaluates the relationship between rainfall
and astronomical/Nakshatra indicators.

The analysis focuses on:

1. Rainfall differences across Gregorian calendar months.
2. Rainfall differences across solar Nakshatras.
3. Comparison of the statistical strength of these groupings.
4. Relationship between Nakshatra and rainfall after accounting for month.
5. Historical comparison between the baseline period (1901–1960)
   and recent period (1961–2025).

The results will provide the statistical foundation for the subsequent
machine-learning stage.

Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats

print("Statistical analysis environment initialized.")

Statistical analysis environment initialized.


Load the datasets

In [2]:
GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "sumitsingh63/EAA_Project/main/data/raw/"
)

IMD_FILE = GITHUB_RAW_BASE + "imd_rainfall_master_1901_2025.csv"
NAKSHATRA_FILE = GITHUB_RAW_BASE + "nakshatra_tags.csv"

imd_df = pd.read_csv(IMD_FILE)
nakshatra_df = pd.read_csv(NAKSHATRA_FILE)

imd_df["Date"] = pd.to_datetime(imd_df["Date"])
nakshatra_df["date"] = pd.to_datetime(nakshatra_df["date"])

master_df = pd.merge(
    imd_df,
    nakshatra_df,
    left_on="Date",
    right_on="date",
    how="inner",
    validate="one_to_one"
)

master_df = master_df.drop(columns=["date"])

print("Master dataset loaded.")
print("Shape:", master_df.shape)

Master dataset loaded.
Shape: (45656, 11)


Create Analysis Variables

In [3]:
master_df["Year"] = master_df["Date"].dt.year
master_df["Month"] = master_df["Date"].dt.month

master_df["Era"] = np.where(
    master_df["Year"] <= 1960,
    "Baseline (1901–1960)",
    "Recent (1961–2025)"
)

print("Analysis variables created.")

Analysis variables created.


Verify the data

In [4]:
print("Observations:", len(master_df))
print("Missing rainfall:", master_df["Rainfall_mm"].isna().sum())
print("Duplicate dates:", master_df["Date"].duplicated().sum())

print(
    "Date range:",
    master_df["Date"].min().date(),
    "to",
    master_df["Date"].max().date()
)

Observations: 45656
Missing rainfall: 0
Duplicate dates: 0
Date range: 1901-01-01 to 2025-12-31


Prepare Groups

We'll use the 27 solar Nakshatras in their astronomical order.

In [5]:
nakshatra_order = [
    "Ashwini", "Bharani", "Krittika", "Rohini",
    "Mrigashira", "Ardra", "Punarvasu", "Pushya",
    "Ashlesha", "Magha", "Purva Phalguni",
    "Uttara Phalguni", "Hasta", "Chitra", "Swati",
    "Vishakha", "Anuradha", "Jyeshtha", "Mula",
    "Purva Ashadha", "Uttara Ashadha", "Shravana",
    "Dhanishta", "Shatabhisha", "Purva Bhadrapada",
    "Uttara Bhadrapada", "Revati"
]

print("Number of solar Nakshatras:",
      master_df["solar_nakshatra"].nunique())

Number of solar Nakshatras: 27
